# EDA Data VCT 2023–2026
Kerangka ini mengikuti delapan butir **Bagian C (Rencana EDA)** di proposal. Jalankan sel dari atas ke bawah.

Cara pakai:
- Letakkan notebook ini di folder yang berisi `vct_2023` ... `vct_2026` (atau ubah `BASE`).
- Setiap bagian punya sel **Catatan kita** di akhirnya. Tulis temuan dan interpretasi kalian sendiri di sana, karena itu yang akan dibaca dosen.

**Penjelasan blok 1:** Blok ini menyiapkan semuanya: memanggil library (pandas untuk tabel, matplotlib untuk grafik), menentukan lokasi data (`BASE`), tahun yang dipakai (`YEARS`), kolom yang menandai satu komposisi (`KEY`), dan fungsi `read` agar file CSV dibaca dengan cara yang sama setiap kali. Tidak ada output, cukup dijalankan sekali di awal.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE = Path(".")                      # folder yang berisi vct_2021 ... vct_2026
YEARS = [2023, 2024, 2025, 2026]      # data utama proposal
KEY = ['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map', 'Team']   # 1 komposisi = 1 tim di 1 map

def read(year, sub, name):
    path = BASE / f"vct_{year}" / sub / name
    return pd.read_csv(path, keep_default_na=False, na_values=[''], low_memory=False)

## C.i Profiling dasar

**Penjelasan blok 2:** Membaca `overview.csv` dari tahun 2023 sampai 2026, menambahkan kolom `Tahun`, lalu menggabungkannya menjadi satu tabel besar bernama `overview`. Output menunjukkan ukuran tabel dan 5 baris pertama untuk melihat isinya.

In [ ]:
frames = []
for y in YEARS:
    o = read(y, "matches", "overview.csv")
    o['Tahun'] = y
    frames.append(o)
overview = pd.concat(frames, ignore_index=True)

print("bentuk data:", overview.shape)
overview.head()

**Penjelasan blok 3:** Melihat struktur tabel: nama kolom, jumlah nilai yang terisi, dan tipe datanya. Kolom yang nilai terisinya lebih sedikit dari jumlah baris berarti punya data kosong.

In [ ]:
overview.info()

**Penjelasan blok 4:** Menghitung berapa kolom untuk tiap tipe data, lalu menampilkan statistik dasar (rata-rata, minimum, maksimum, dan sebagainya) untuk kolom angka sebagai gambaran umum isi datanya.

In [ ]:
print(overview.dtypes.value_counts())
overview.describe().T.head(10)

**Penjelasan blok 5:** Mencari 10 kolom dengan data kosong terbanyak. Kekosongan di kolom statistik pemain (Rating, Kills, dan lain-lain) wajar menurut Data Card. Yang penting, kolom yang kita pakai (Map, Team, Agents) tidak kosong.

In [ ]:
# 10 kolom dengan nilai kosong terbanyak
overview.isna().sum().sort_values(ascending=False).head(10)

### Membentuk tabel komposisi
Dari `overview.csv` kita ambil baris `Side == 'both'` dan buang baris ringkasan `All Maps`. Satu komposisi = satu tim pada satu map, dan dianggap valid bila berisi tepat 5 pemain.

**Penjelasan blok 6:** Blok terpenting di bagian awal. Tabel `overview` berisi satu baris per pemain plus baris ringkasan, jadi di sini disaring (hanya `Side = both` dan bukan `All Maps`), lalu dikelompokkan menjadi satu baris per komposisi (satu tim di satu map). Komposisi yang jumlah pemainnya bukan 5 dibuang. Hasilnya dua tabel: `comp` (satu baris per komposisi) dan `pick` (satu baris per pemain/agent). Baris `assert` memastikan jumlahnya konsisten. **Cara baca:** angka "komposisi valid" adalah jumlah data yang kita punya.

In [ ]:
pemain = overview[(overview['Side'] == 'both') & (overview['Map'] != 'All Maps')].copy()
pemain['Agents'] = pemain['Agents'].astype(str)

comp = (pemain.groupby(KEY + ['Tahun'])
        .agg(jumlah_pemain=('Player', 'size'), agents=('Agents', list))
        .reset_index())

print("semua komposisi:", len(comp))
print(comp['jumlah_pemain'].value_counts())

n_tidak_valid = int((comp['jumlah_pemain'] != 5).sum())
comp = comp[comp['jumlah_pemain'] == 5].reset_index(drop=True)   # hanya komposisi valid
pick = pemain.merge(comp[KEY], on=KEY)                           # 1 baris = 1 pemain/agent pada komposisi valid

print("komposisi valid:", len(comp), "| baris pemain:", len(pick))
assert len(pick) == len(comp) * 5

**Catatan kita (C.i):** ...

## C.ii Distribusi kelas (frekuensi pemilihan tiap agent)

**Penjelasan blok 7:** Menghitung seberapa sering tiap agent dipilih, lalu menampilkannya sebagai tabel dan diagram batang. Di akhir ada rasio agent terbanyak dibanding tersedikit untuk melihat seberapa tidak seimbang kelasnya. Kolom `persen` adalah porsi dari seluruh pilihan.

In [ ]:
dist = pick['Agents'].value_counts()
tabel = pd.DataFrame({'jumlah': dist, 'persen': (dist / dist.sum() * 100).round(1)})
print(tabel.head(10))

dist.plot(kind='bar', figsize=(12, 4), title='Frekuensi pemilihan agent 2023-2026')
plt.ylabel('jumlah dipilih'); plt.tight_layout(); plt.show()

print("jumlah agent:", len(dist))
print("rasio agent terbanyak : tersedikit =", round(dist.max() / dist.min(), 1))

**Catatan kita (C.ii):** apakah kelasnya seimbang? agent mana yang sangat jarang dipilih, dan kenapa (misalnya agent baru)?

## C.iii Distribusi map

**Penjelasan blok 8:** Menghitung berapa kali tiap map dimainkan. Satu map dimainkan dua tim, jadi duplikatnya dibuang dulu agar tiap map hanya dihitung sekali.

In [ ]:
game = comp.drop_duplicates(['Tournament', 'Stage', 'Match Type', 'Match Name', 'Map'])   # 1 baris = 1 map yang dimainkan
map_count = game['Map'].value_counts()
print(map_count)

map_count.plot(kind='bar', figsize=(8, 3.5), title='Jumlah map yang dimainkan 2023-2026')
plt.tight_layout(); plt.show()

**Catatan kita (C.iii):** map mana yang paling sering dan paling jarang dimainkan? apakah ada map yang jumlah datanya sangat sedikit?

## C.iv Agent per map (pick rate dan win rate)
**Pick rate** = persentase komposisi di map itu yang memuat agent tersebut.

**Penjelasan blok 9:** Menghitung pick rate tiap agent di tiap map (porsi komposisi di map itu yang memuat agent tersebut), lalu menggambarnya sebagai peta warna. Warna lebih terang berarti agent itu lebih sering dipilih di map tersebut.

In [ ]:
per_map = pick.groupby(['Map', 'Agents']).size().rename('jumlah').reset_index()
total_map = comp.groupby('Map').size().rename('total_komposisi')
per_map = per_map.merge(total_map, on='Map')
per_map['pick_rate'] = per_map['jumlah'] / per_map['total_komposisi']

pr = per_map.pivot(index='Agents', columns='Map', values='pick_rate').fillna(0)

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(pr.values, aspect='auto')
ax.set_xticks(range(len(pr.columns))); ax.set_xticklabels(pr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pr.index))); ax.set_yticklabels(pr.index)
ax.set_title('Pick rate agent per map (2023-2026)')
fig.colorbar(im); plt.tight_layout(); plt.show()

**Win rate** dihitung dari `teams_picked_agents.csv`. File ini berisi baris rekap (`All Stages`, `All Match Types`) yang harus dibuang, kalau tidak jumlahnya terhitung dua kali.

**Penjelasan blok 10:** Menghitung win rate tiap agent di tiap map dari `teams_picked_agents.csv`. Baris rekap dibuang dulu supaya pertandingan tidak terhitung ganda, lalu jumlah menang dan kalah dijumlahkan per agent per map. `MIN_MATCH` adalah batas minimum jumlah dimainkan agar win rate dianggap layak. **Cara baca:** waspadai win rate tinggi yang kolom `dimainkan`-nya kecil.

In [ ]:
tpa = pd.concat([read(y, "agents", "teams_picked_agents.csv") for y in YEARS], ignore_index=True)
print("baris sebelum filter:", len(tpa), "| total Maps Played:", tpa['Total Maps Played'].sum())

tpa = tpa[(tpa['Stage'] != 'All Stages') & (tpa['Match Type'] != 'All Match Types') & (tpa['Map'] != 'All Maps')]
print("baris sesudah filter:", len(tpa), "| total Maps Played:", tpa['Total Maps Played'].sum())
print("pembanding: jumlah baris pemain dari overview =", len(pick))   # harus mendekati

wr = tpa.groupby(['Map', 'Agent'])[['Total Wins By Map', 'Total Loss By Map']].sum()
wr['dimainkan'] = wr.sum(axis=1)
wr['win_rate'] = wr['Total Wins By Map'] / wr['dimainkan']
wr = wr.reset_index()

MIN_MATCH = 10      # batas minimum sampel; boleh diubah
print("kombinasi agent-map:", len(wr), "| yang dimainkan >=", MIN_MATCH, "kali:", (wr['dimainkan'] >= MIN_MATCH).sum())
wr[wr['dimainkan'] >= MIN_MATCH].sort_values('win_rate', ascending=False).head(10)

**Penjelasan blok 11:** Peta warna win rate agent per map, hanya untuk kombinasi yang dimainkan sedikitnya `MIN_MATCH` kali. Sel kosong berarti datanya belum cukup.

In [ ]:
wr_ok = wr[wr['dimainkan'] >= MIN_MATCH]
wr_pivot = wr_ok.pivot(index='Agent', columns='Map', values='win_rate')

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(wr_pivot.values, aspect='auto')
ax.set_xticks(range(len(wr_pivot.columns))); ax.set_xticklabels(wr_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(wr_pivot.index))); ax.set_yticklabels(wr_pivot.index)
ax.set_title(f'Win rate agent per map (minimal {MIN_MATCH} kali dimainkan)')
fig.colorbar(im); plt.tight_layout(); plt.show()

**Catatan kita (C.iv):** agent mana yang dominan di map tertentu? apakah win rate yang tinggi didukung sampel yang cukup banyak?

## C.v Peran agent
Dataset **tidak punya kolom peran**, jadi tabelnya dibuat manual. Isian di bawah adalah tebakan awal.
**Wajib diverifikasi dari sumber resmi Valorant** sebelum dipakai. Agent yang belum diketahui perannya dibiarkan `None` (isi sendiri).

**Penjelasan blok 12:** Tabel peran agent (Duelist, Initiator, Controller, Sentinel) dibuat manual karena dataset tidak punya kolom peran. Output mengecek agent yang belum punya peran. Isinya wajib diverifikasi dari sumber resmi Valorant.

In [ ]:
ROLE = {
    # Duelist
    'jett': 'Duelist', 'phoenix': 'Duelist', 'reyna': 'Duelist', 'raze': 'Duelist',
    'yoru': 'Duelist', 'neon': 'Duelist', 'iso': 'Duelist', 'waylay': 'Duelist',
    # Initiator
    'sova': 'Initiator', 'breach': 'Initiator', 'skye': 'Initiator', 'kayo': 'Initiator',
    'fade': 'Initiator', 'gekko': 'Initiator', 'tejo': 'Initiator',
    # Controller
    'brimstone': 'Controller', 'omen': 'Controller', 'viper': 'Controller',
    'astra': 'Controller', 'harbor': 'Controller', 'clove': 'Controller',
    # Sentinel
    'sage': 'Sentinel', 'cypher': 'Sentinel', 'killjoy': 'Sentinel',
    'chamber': 'Sentinel', 'deadlock': 'Sentinel', 'vyse': 'Sentinel',
    # agent baru: isi sendiri setelah dicek
    'miks': None, 'veto': None,
}

agents_data = sorted(pick['Agents'].unique())
print("agent di data:", len(agents_data))
print("agent yang belum punya peran:", [a for a in agents_data if ROLE.get(a) is None])

**Penjelasan blok 13:** Menghitung ada berapa agent dari tiap peran di setiap komposisi, lalu melihat polanya, misalnya berapa persen komposisi yang punya 1 atau 2 Controller. Dari sini kita menentukan aturan keseimbangan peran untuk sistem rekomendasi.

In [ ]:
pick['Peran'] = pick['Agents'].map(ROLE)

# hanya komposisi yang kelima agent-nya sudah punya peran
lengkap = pick.groupby(KEY)['Peran'].apply(lambda s: s.notna().all())
role_count = pick.groupby(KEY)['Peran'].value_counts().unstack(fill_value=0)
role_count = role_count[lengkap.reindex(role_count.index).fillna(False)]
print("komposisi yang dipakai:", len(role_count), "dari", len(comp))

for p in role_count.columns:
    print(p, "-> persentase komposisi menurut jumlah:", role_count[p].value_counts(normalize=True).sort_index().round(3).to_dict())

print()
print("komposisi dengan minimal 1 Controller:", round((role_count['Controller'] >= 1).mean() * 100, 1), "%")
print("komposisi dengan >= 3 Duelist       :", round((role_count['Duelist'] >= 3).mean() * 100, 1), "%")

**Catatan kita (C.v):** dari angka di atas, apa batas jumlah tiap peran yang masuk akal untuk aturan keseimbangan peran? (misalnya maksimal berapa Duelist, minimal berapa Controller)

## C.vi Tren waktu (pergeseran meta)

**Penjelasan blok 14:** Menampilkan 5 agent teratas tiap tahun dan menggambar garis pick rate 8 agent terpopuler per tahun untuk melihat pergeseran meta. Angkanya adalah porsi komposisi yang memuat agent, jadi berbeda dengan kolom `persen` di bagian distribusi agent.

In [ ]:
per_tahun = pick.groupby(['Tahun', 'Agents']).size().rename('jumlah').reset_index()
total_tahun = comp.groupby('Tahun').size().rename('total_komposisi')
per_tahun = per_tahun.merge(total_tahun, on='Tahun')
per_tahun['pick_rate'] = per_tahun['jumlah'] / per_tahun['total_komposisi']

for y in YEARS:
    top = per_tahun[per_tahun['Tahun'] == y].nlargest(5, 'pick_rate')
    print(y, [(a, round(p * 100, 1)) for a, p in zip(top['Agents'], top['pick_rate'])])

tren = per_tahun.pivot(index='Tahun', columns='Agents', values='pick_rate').fillna(0)
tren[dist.head(8).index].plot(figsize=(9, 4), marker='o', title='Pick rate 8 agent terpopuler per tahun')
plt.ylabel('pick rate'); plt.tight_layout(); plt.show()

print("jumlah agent per tahun:", per_tahun.groupby('Tahun')['Agents'].nunique().to_dict())

### Tren 2021–2026 (tambahan)
Proposal merencanakan data 2021–2022 untuk analisis tren. Bagian ini memakai `teams_picked_agents.csv` (lebih ringan daripada `overview.csv`) dengan filter rekap yang sama.

**Penjelasan blok 15:** Sama seperti blok sebelumnya tetapi mencakup 2021 sampai 2026, memakai `teams_picked_agents.csv` yang lebih ringan. Output menunjukkan jumlah agent dan tiga agent teratas di setiap tahun.

In [ ]:
ALL_YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
hasil = {}
for y in ALL_YEARS:
    t = read(y, "agents", "teams_picked_agents.csv")
    t = t[(t['Stage'] != 'All Stages') & (t['Match Type'] != 'All Match Types') & (t['Map'] != 'All Maps')]
    pilih = t.groupby('Agent')['Total Maps Played'].sum()
    hasil[y] = pilih / pilih.sum()

tren_all = pd.DataFrame(hasil).fillna(0)
for y in ALL_YEARS:
    top = tren_all[y].nlargest(3)
    print(y, "| jumlah agent:", int((tren_all[y] > 0).sum()), "| tiga teratas:", [(a, round(p * 100, 1)) for a, p in top.items()])

tren_all.loc[tren_all.mean(axis=1).nlargest(8).index].T.plot(figsize=(9, 4), marker='o', title='Porsi pemilihan 8 agent terpopuler, 2021-2026')
plt.ylabel('porsi pemilihan'); plt.tight_layout(); plt.show()

**Catatan kita (C.vi):** agent mana yang naik atau turun? bagaimana pengaruhnya terhadap rencana pembagian training/testing berdasarkan tahun?

## C.vii Kualitas data

**Penjelasan blok 16:** Memeriksa masalah kualitas data: baris duplikat, pemain bernama `nan`, tim `TBD`, komposisi bukan 5 pemain, serta sel Agents yang berisi lebih dari satu agent atau kosong. Nilai 0 berarti masalah itu tidak ada pada data 2023–2026.

In [ ]:
print("baris duplikat di overview   :", overview.duplicated().sum())
print("pemain bernama 'nan'          :", (overview['Player'] == 'nan').sum())
print("baris tim 'TBD'               :", (overview['Team'] == 'TBD').sum())
print("komposisi tidak berisi 5 pemain:", n_tidak_valid)
print("sel Agents berisi >1 agent    :", pemain['Agents'].str.contains(',').sum())
print("Agents kosong                 :", (pemain['Agents'] == 'nan').sum())

**Penjelasan blok 17:** Melihat sebaran jumlah sampel per kombinasi agent–map dan berapa kombinasi yang di bawah `MIN_MATCH`, untuk menilai seberapa bisa dipercaya win rate-nya.

In [ ]:
# sampel per kombinasi agent-map (untuk menilai apakah win rate bisa dipercaya)
sampel = wr['dimainkan']
print(sampel.describe().round(1))
print(f"kombinasi dengan sampel < {MIN_MATCH}: {(sampel < MIN_MATCH).sum()} dari {len(sampel)}")

**Catatan kita (C.vii):** masalah data apa yang masih ada, dan bagaimana kita menanganinya di praproses?

## C.viii Visualisasi

Bar chart (agent dan map) dan heatmap (pick rate dan win rate per map) sudah ada di bagian C.ii sampai C.iv. Di bawah ini histogram jumlah sampel per kombinasi agent-map.

**Penjelasan blok 18:** Histogram sebaran jumlah sampel per kombinasi agent–map, dengan garis merah di batas minimum. Banyak batang di kiri garis berarti banyak kombinasi yang datanya sedikit.

In [ ]:
wr['dimainkan'].plot(kind='hist', bins=30, figsize=(8, 3.5), title='Sebaran jumlah sampel per kombinasi agent-map')
plt.axvline(MIN_MATCH, color='red', linestyle='--', label=f'batas minimum = {MIN_MATCH}')
plt.xlabel('jumlah kali dimainkan'); plt.legend(); plt.tight_layout(); plt.show()

**Catatan kita (C.viii):** ...

## Menyimpan hasil
Tabel komposisi bersih ini akan dipakai di praproses dan model. Folder `hasil/` sebaiknya masuk `.gitignore` bila ukurannya besar.

**Penjelasan blok 19:** Menyimpan tabel komposisi bersih dan tabel win rate ke folder `hasil/` dalam bentuk CSV, supaya notebook praproses dan model bisa memakainya tanpa mengulang semua langkah di atas.

In [ ]:
Path("hasil").mkdir(exist_ok=True)

simpan = comp.copy()
simpan['agents'] = simpan['agents'].apply(lambda l: '|'.join(l))
simpan.to_csv("hasil/komposisi_valid_2023_2026.csv", index=False)
wr.to_csv("hasil/winrate_agent_per_map.csv", index=False)
print("tersimpan:", list(Path("hasil").glob("*.csv")))

## Kesimpulan EDA
Tulis dengan kata kalian sendiri:
1. Seberapa tidak seimbang kelas agent, dan apa dampaknya pada model?
2. Map dan agent mana yang paling berpengaruh?
3. Aturan keseimbangan peran apa yang kita ambil dari data?
4. Apa keputusan cakupan data akhir (tahun yang dipakai)?